# LAB 10 — TOKENIZATION, OFFSET MAPPING VÀ ANSWER SPAN ALIGNMENT CHO EXTRACTIVE QA

**Môn học**: Xử lý ngôn ngữ tự nhiên và ứng dụng  
**Học kỳ**: Học kỳ I - 2026  
**Chủ đề**: From Token Classification to Span Prediction  
**Sinh viên thực hiện**: Nguyễn Văn Thành  
**Mục tiêu**: Triển khai và kiểm chứng chi tiết pipeline tokenization `(question, context)`, kiểm tra `token_type_ids` giữa BERT và DistilBERT, triển khai Sliding Window (`stride`, `return_overflowing_tokens`, `return_offsets_mapping`), và xử lý 4 trường hợp biên khi ánh xạ `character span -> token span` trên tập SQuAD.

## 1. Khởi Tạo Tokenizer và Kiểm Tra Cấu Trúc Đầu Vào QA
So sánh cấu trúc đầu vào giữa `bert-base-uncased` (có `token_type_ids`) và `distilbert-base-uncased` (không sử dụng `token_type_ids`).

In [1]:
from transformers import AutoTokenizer
from datasets import load_dataset
import pandas as pd

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
distilbert_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

question = "What is the capital of France?"
context = "The capital of France is Paris."

bert_inputs = bert_tokenizer(question, context)
distil_inputs = distilbert_tokenizer(question, context)

print("BERT keys:", list(bert_inputs.keys()))
print("DistilBERT keys:", list(distil_inputs.keys()))
print("BERT tokens:", bert_tokenizer.convert_ids_to_tokens(bert_inputs["input_ids"]))
print("BERT token_type_ids:", bert_inputs["token_type_ids"])
print("BERT attention_mask:", bert_inputs["attention_mask"])

C:\Users\Thanh\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BERT keys: ['input_ids', 'token_type_ids', 'attention_mask']
DistilBERT keys: ['input_ids', 'token_type_ids', 'attention_mask']
BERT tokens: ['[CLS]', 'what', 'is', 'the', 'capital', 'of', 'france', '?', '[SEP]', 'the', 'capital', 'of', 'france', 'is', 'paris', '.', '[SEP]']
BERT token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1]
BERT attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## 2. Phân Biệt Character Index, Word Index và Token Index (Bài Toán 3)
Kiểm chứng lại kết quả tính tay cho câu `"The capital of France is Paris."` với đáp án `"Paris"`.

In [2]:
answer_text = "Paris"
answer_start = context.index(answer_text)
answer_end = answer_start + len(answer_text)

words = context.split()
word_start_idx = [i for i, w in enumerate(words) if "Paris" in w][0]

encoding_context = distilbert_tokenizer(context, return_offsets_mapping=True)
tokens_ctx = distilbert_tokenizer.convert_ids_to_tokens(encoding_context["input_ids"])
offsets_ctx = encoding_context["offset_mapping"]

token_start = None
token_end = None
for idx, (s, e) in enumerate(offsets_ctx):
    if s == 0 and e == 0:
        continue
    if s <= answer_start < e:
        token_start = idx
    if s < answer_end <= e:
        token_end = idx

print(f"Character span: [{answer_start}, {answer_end}) -> '{context[answer_start:answer_end]}'")
print(f"Word index: {word_start_idx} -> '{words[word_start_idx]}'")
print(f"Token span (context-only with [CLS]): start_token={token_start}, end_token={token_end}")

df_offsets = pd.DataFrame({
    "Token Index": list(range(len(tokens_ctx))),
    "Token": tokens_ctx,
    "Offset Start": [o[0] for o in offsets_ctx],
    "Offset End": [o[1] for o in offsets_ctx],
    "Matched Substring": [context[o[0]:o[1]] if o != (0, 0) else "<SPECIAL>" for o in offsets_ctx]
})
print(df_offsets.to_string(index=False))

Character span: [25, 30) -> 'Paris'
Word index: 5 -> 'Paris.'
Token span (context-only with [CLS]): start_token=6, end_token=6
 Token Index   Token  Offset Start  Offset End Matched Substring
           0   [CLS]             0           0         <SPECIAL>
           1     the             0           3               The
           2 capital             4          11           capital
           3      of            12          14                of
           4  france            15          21            France
           5      is            22          24                is
           6   paris            25          30             Paris
           7       .            30          31                 .
           8   [SEP]             0           0         <SPECIAL>


## 3. Core Implementation: Sliding Window, Stride và Overflow Mapping (Mục 15)
Áp dụng đầy đủ pipeline với `truncation="only_second"`, `max_length`, `stride`, `return_overflowing_tokens=True`, `return_offsets_mapping=True` trên một mẫu dài thực tế từ tập **SQuAD**.

In [3]:
squad = load_dataset("rajpurkar/squad")
long_sample = None
for item in squad["validation"]:
    if len(item["context"]) > 1100 and item["answers"]["answer_start"][0] > 700:
        long_sample = item
        break

q_long = long_sample["question"]
c_long = long_sample["context"]
ans_text = long_sample["answers"]["text"][0]
ans_start = long_sample["answers"]["answer_start"][0]
ans_end = ans_start + len(ans_text)

print("Question:", q_long)
print("Context length (chars):", len(c_long))
print(f"Gold Answer: '{ans_text}' at char span [{ans_start}, {ans_end})")

inputs = distilbert_tokenizer(
    [q_long],
    [c_long],
    truncation="only_second",
    max_length=128,
    stride=32,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding="max_length"
)

print("Keys returned:", list(inputs.keys()))
print("Number of windows generated:", len(inputs["input_ids"]))
print("overflow_to_sample_mapping:", inputs["overflow_to_sample_mapping"])
print("Window 0 input_ids length:", len(inputs["input_ids"][0]))
print("Window 0 attention_mask sum (active tokens):", sum(inputs["attention_mask"][0]))

Question: Which Panthers RB scored 6 TDs in the 13 games leading up to Super Bowl 50?
Context length (chars): 1271
Gold Answer: 'Jonathan Stewart' at char span [964, 980)
Keys returned: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'overflow_to_sample_mapping']
Number of windows generated: 4
overflow_to_sample_mapping: [0, 0, 0, 0]
Window 0 input_ids length: 128
Window 0 attention_mask sum (active tokens): 128


## 4. Triển Khai Thuật Toán Answer Span Alignment (Mục 16)
Ánh xạ `answer_start`, `answer_end` sang `(start_positions, end_positions)` trên từng window dựa trên `sequence_ids` và `offset_mapping`.

In [4]:
def align_answer_to_window(offsets, sequence_ids, ans_start, ans_end):
    context_start = 0
    while context_start < len(sequence_ids) and sequence_ids[context_start] != 1:
        context_start += 1
    context_end = len(sequence_ids) - 1
    while context_end >= 0 and sequence_ids[context_end] != 1:
        context_end -= 1

    if context_start >= len(sequence_ids) or context_end < 0:
        return 0, 0, "NO_CONTEXT"

    if offsets[context_start][0] > ans_start or offsets[context_end][1] < ans_end:
        return 0, 0, "OUTSIDE_WINDOW"

    idx = context_start
    while idx <= context_end and offsets[idx][0] <= ans_start:
        idx += 1
    start_token = idx - 1

    idx = context_end
    while idx >= context_start and offsets[idx][1] >= ans_end:
        idx -= 1
    end_token = idx + 1

    return start_token, end_token, "IN_WINDOW"

window_records = []
for win_idx in range(len(inputs["input_ids"])):
    offsets = inputs["offset_mapping"][win_idx]
    seq_ids = inputs.sequence_ids(win_idx)
    ctx_indices = [i for i, s in enumerate(seq_ids) if s == 1]
    c_char_range = (offsets[ctx_indices[0]][0], offsets[ctx_indices[-1]][1]) if ctx_indices else (0, 0)
    s_tok, e_tok, status = align_answer_to_window(offsets, seq_ids, ans_start, ans_end)
    decoded_span = ""
    if status == "IN_WINDOW":
        toks = distilbert_tokenizer.convert_ids_to_tokens(inputs["input_ids"][win_idx][s_tok:e_tok + 1])
        decoded_span = " ".join(toks)
    window_records.append({
        "Window": win_idx,
        "Sample Mapping": inputs["overflow_to_sample_mapping"][win_idx],
        "Context Char Range": str(c_char_range),
        "Status": status,
        "Start Token": s_tok,
        "End Token": e_tok,
        "Aligned Tokens": decoded_span
    })

print(pd.DataFrame(window_records).to_string(index=False))

 Window  Sample Mapping Context Char Range         Status  Start Token  End Token   Aligned Tokens
      0               0           (0, 468) OUTSIDE_WINDOW            0          0                 
      1               0         (327, 785) OUTSIDE_WINDOW            0          0                 
      2               0        (646, 1135)      IN_WINDOW           91         92 jonathan stewart
      3               0        (986, 1271) OUTSIDE_WINDOW            0          0                 


## 5. Kiểm Tra 4 Trường Hợp Biên Bắt Buộc (Edge Cases — Mục 16)
Kiểm tra:
1. Answer bắt đầu giữa token.
2. Answer kết thúc giữa token.
3. Answer bị truncation khi không dùng `return_overflowing_tokens`.
4. Answer nằm ngoài window hiện tại khi dùng sliding window.

In [5]:
def test_span_edge_case(q_text, c_text, ans_start_char, ans_end_char, max_len=384, use_overflow=True):
    enc = distilbert_tokenizer(
        q_text,
        c_text,
        truncation="only_second",
        max_length=max_len,
        stride=32 if use_overflow else 0,
        return_overflowing_tokens=use_overflow,
        return_offsets_mapping=True
    )
    if use_overflow:
        results = []
        for w in range(len(enc["input_ids"])):
            s, e, st = align_answer_to_window(enc["offset_mapping"][w], enc.sequence_ids(w), ans_start_char, ans_end_char)
            toks = distilbert_tokenizer.convert_ids_to_tokens(enc["input_ids"][w][s:e+1]) if st == "IN_WINDOW" else ["[CLS]"]
            results.append((w, st, s, e, toks))
        return results
    else:
        s, e, st = align_answer_to_window(enc["offset_mapping"], enc.sequence_ids(0), ans_start_char, ans_end_char)
        toks = distilbert_tokenizer.convert_ids_to_tokens(enc["input_ids"][s:e+1]) if st == "IN_WINDOW" else ["[CLS]"]
        return [(0, st, s, e, toks)]

edge_ctx = "Supercalifragilisticexpialidocious is an extraordinarily long English word."
edge_q = "What substring appears inside the first word?"

mid_start = edge_ctx.index("fragil")
mid_start_end = mid_start + len("fragilistic")
res_case1 = test_span_edge_case(edge_q, edge_ctx, mid_start, mid_start_end)

mid_end_start = edge_ctx.index("Super")
mid_end_end = mid_end_start + len("Supercalifrag")
res_case2 = test_span_edge_case(edge_q, edge_ctx, mid_end_start, mid_end_end)

res_case3 = test_span_edge_case(q_long, c_long, ans_start, ans_end, max_len=64, use_overflow=False)
res_case4 = test_span_edge_case(q_long, c_long, ans_start, ans_end, max_len=128, use_overflow=True)

print("Case 1 (Starts mid-token):", res_case1)
print("Case 2 (Ends mid-token):", res_case2)
print("Case 3 (Truncated without overflow, max_len=64):", res_case3)
print("Case 4 (Outside window 0, found in later window):", res_case4)

Case 1 (Starts mid-token): [(0, 'IN_WINDOW', 14, 17, ['##if', '##rag', '##ilis', '##tic'])]
Case 2 (Ends mid-token): [(0, 'IN_WINDOW', 12, 15, ['super', '##cal', '##if', '##rag'])]
Case 3 (Truncated without overflow, max_len=64): [(0, 'OUTSIDE_WINDOW', 0, 0, ['[CLS]'])]
Case 4 (Outside window 0, found in later window): [(0, 'OUTSIDE_WINDOW', 0, 0, ['[CLS]']), (1, 'OUTSIDE_WINDOW', 0, 0, ['[CLS]']), (2, 'IN_WINDOW', 91, 92, ['jonathan', 'stewart']), (3, 'OUTSIDE_WINDOW', 0, 0, ['[CLS]'])]
